# Human–Robot Handover with Reinforcement Learning

**What you will learn:**
1. What MuJoCo and robosuite are, and how the environment works
2. How to read observations (robot state, object state, camera images)
3. How to apply actions and control the robot
4. How to define and customize reward functions
5. How to train a reinforcement learning (RL) policy
6. How to evaluate and visualize a trained policy

**The Task:** One robot arm starts already holding a hammer and hands it to a person. The handle center must stay inside 2 cm of the palm-sphere center for 3 seconds, no faster than 5 mm/s, without touching the person.

| Section | Topic |
|---|---|
| Part 0 | Setup |
| Part 1 | Creating the environment |
| Part 2 | Observations |
| Part 3 | Actions |
| Part 4 | Reward functions |
| Part 5 | Training an RL policy |
| Part 6 | Evaluation and visualization |

**Rules — read before you start**

1. **Run the cells in order, one at a time, from top to bottom.** Do not skip a cell, and do not jump back to rerun earlier cells. After fixing your own answer you may rerun that answer cell. If anything else goes wrong, restart the kernel and run again from the first cell.
2. **Only write in the answer areas.** In code cells, write only between `# --- write your answer from here ---` (in Question 6: `# --- reward: write from here ---`) and the dashed `# -----------------------------------` line below it. In markdown, write only in the cells that ask for your sentences (`Your sentence:` and the list of keys in Question 4). Do not change anything else. **If anything outside the answer areas is changed, the question it belongs to scores 0.**
3. **Submit only this notebook (`.ipynb`)**, saved after Part 6 has finished. Only the outputs saved in this notebook are graded: a question whose cell has no output scores 0.
4. **Feel free to use any AI tools** (for example ChatGPT, Claude, Copilot, or Cursor) to help you understand the code and write your answers. **Warning:** AI tools often edit more than you asked. Check every change, and do not let the AI touch anything outside the answer areas: any change there makes that question score 0, as in rule 2.

---


## Part 0 — Setup

Follow `README.md` in this folder to install everything. Then choose the `courseproj` kernel and run the next cell to check the imports.

In [ ]:
import os
import sys
from pathlib import Path

# Windows only. Mac / Linux skip this.
if sys.platform == "win32":
    os.environ["MUJOCO_GL"] = "wgl"
    Path(r"C:\tmp").mkdir(exist_ok=True)

import cv2
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display

import gymnasium as gym
from gymnasium import spaces
import robosuite as suite
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

print("robosuite version:", suite.__version__)
print("numpy version:", np.__version__)


---
## Part 1 — Creating the Environment

### 1.1 What is MuJoCo?

**MuJoCo** (Multi-Joint dynamics with Contact) is a physics simulator. It models rigid bodies, joints, contacts, and actuators.

**robosuite** is a robotics library built on top of MuJoCo. It provides:
- Ready-made robot arms (Panda, XArm7, UR5e, ...)
- Ready-made manipulation tasks (Lift, Pick-and-Place, Assembly, ...)
- Cameras, sensors, and a standard observation / action interface

The interface follows the classic **Gym loop**:

```
obs = env.reset()                        # start a new episode
obs, reward, done, info = env.step(action)   # advance one timestep
```

### 1.2 The human-robot handover task

```
  Robot  ——  table  ——  person
           [hammer already in the gripper]
           [cyan palm sphere above the palm]
```

- One **XArm7** starts with the hammer head already pinched.
- A **person** stands on the far side of the table, facing the robot, right palm up.
- The cyan **palm sphere** is the target (5 cm radius). Its center sits 6 cm out from the hand along the palm normal.
- **Goal**: keep the handle center within **2 cm** of the sphere center for **3 seconds** (60 steps at 20 Hz), at **5 mm/s** or slower.
- Touching the person, including the hand, ends the episode on that step.

### 1.3 Creating the environment


In [ ]:
env = suite.make(
    env_name="HumanRobotHandover",  # which task to load
    robots="XArm7Handover",         # one XArm7

    use_camera_obs=True,            # include camera images in observations
    camera_names=[
        "agentview",
        "birdview",
        "robot0_eye_in_hand",
        "extra_side_plusx",
        "extra_palm_close",
    ],

    reward_shaping=False,           # sparse reward for now (1 on success, 0 otherwise)
    horizon=300,                    # max timesteps per episode (15 s at 20 Hz)
    control_freq=20,                # 20 Hz control (50 ms per step)
    seed=0,
    hard_reset=False,
)
# The episode starts with the hammer already in the gripper.
env.start_holding = True
obs = env.reset()

print("Environment created successfully!")
print("Type:", type(env).__name__)

print("Cameras in this model:", [env.sim.model.camera_id2name(i) for i in range(env.sim.model.ncam)])


The comments on each argument say what that line does. The same knobs show up again in Part 5, when you build the training environment.


---
## Part 2 — Observations

After `env.reset()` or `env.step(action)`, you get an **observation dictionary**. Each key is a numpy array.

### 2.1 Listing all observation keys

| Key | What it does |
|---|---|
| `robot0_joint_pos` | 7 joint angles (radians) |
| `robot0_joint_pos_cos` | Cosine of the 7 joint angles |
| `robot0_joint_pos_sin` | Sine of the 7 joint angles |
| `robot0_joint_vel` | 7 joint velocities (rad/s) |
| `robot0_joint_acc` | 7 joint accelerations (rad/s²) |
| `robot0_eef_pos` | End-effector XYZ position (meters) |
| `robot0_eef_quat` | End-effector orientation, quaternion `[x, y, z, w]` |
| `robot0_eef_quat_site` | End-effector site orientation, quaternion |
| `robot0_gripper_qpos` | Gripper finger positions |
| `robot0_gripper_qvel` | Gripper finger velocities |
| `handle_pos` | Hammer handle center XYZ (meters) |
| `head_pos` | Hammer head XYZ (meters) |
| `palm_pos` | Right wrist mesh XYZ, not the palm sphere (meters) |
| `hammer_quat` | Hammer orientation, quaternion |
| `hammer_vel` | Hammer linear velocity (m/s) |
| `agentview_image` | Third-person camera image, shape `(256, 256, 3)` |
| `birdview_image` | Top-down camera image, shape `(256, 256, 3)` |
| `robot0_eye_in_hand_image` | Wrist camera image, shape `(256, 256, 3)` |
| `extra_side_plusx_image` | Side camera from +x, shape `(256, 256, 3)` |
| `extra_palm_close_image` | Close-up of the palm sphere, shape `(256, 256, 3)` |
| `robot0_proprio-state` | Concatenated robot proprioception vector |
| `object-state` | Concatenated object-state vector |


### 2.2 Robot proprioception


## Question 1 — Robot observations
**6 of 14 pts**

Use the example below as a reference and write your own answer in the next cell.

| Item | Pts |
|---|---|
| `obs["robot0_joint_vel"]` | 2 |
| `obs["robot0_eef_quat"]` | 2 |
| `obs["robot0_gripper_qpos"]` | 2 |


In [ ]:
# ========== example ==========
print("EEF position (m):      ", obs["robot0_eef_pos"])
print("Joint positions (rad): ", obs["robot0_joint_pos"])
# ==========================


In [ ]:
# --- write your answer from here ---

# -----------------------------------


### 2.3 Object state


## Question 1 (continued) — Object observations
**8 of 14 pts**

Use the example below as a reference and write your own answer in the next cell.

| Item | Pts |
|---|---|
| `obs["head_pos"]` | 2 |
| `obs["palm_pos"]` | 2 |
| `obs["hammer_quat"]` | 2 |
| `obs["hammer_vel"]` | 2 |


In [ ]:
# ========== example ==========
print("Handle position (m):   ", obs["handle_pos"])
# ==========================


In [ ]:
# --- write your answer from here ---

# -----------------------------------


### 2.4 Camera images

robosuite stores images upside-down. Apply `[::-1]` before displaying or saving.


## Question 2 — Camera images
**12 pts**

Use the example below as a reference and write your own answer in the next cell. Keep `[::-1]` so the image is right-side up. 3 pts per image; a picture that is still upside-down loses 1 of the 3.

| Item | Pts |
|---|---|
| `birdview` | 3 |
| `robot0_eye_in_hand` | 3 |
| `extra_side_plusx` | 3 |
| `extra_palm_close` | 3 |


In [ ]:
img = obs["agentview_image"]
print(f"agentview_image: shape={img.shape}, dtype={img.dtype}")
print("Cameras in the model:", [env.sim.model.camera_id2name(i) for i in range(env.sim.model.ncam)])


In [ ]:
# ========== example ==========
plt.imshow(obs["agentview_image"][::-1])
plt.axis("off")
plt.title("agentview")
plt.show()
# ==========================


In [ ]:
# --- write your answer from here ---

# -----------------------------------


---
## Part 3 — Actions

### 3.1 Understanding the action space

Actions are a flat numpy array. This arm uses an operational-space controller, so the action is a pose increment plus the gripper:

```
action = [ dx, dy, dz, droll, dpitch, dyaw, gripper ]
                                              total = 7 dimensions
```

- Commands are in `[-1, 1]`. A full position command moves the end-effector **2 cm**. A full rotation command turns it **0.02 rad**. Both apply over one 20 Hz step.
- **Gripper command**: `+1.0` = close, `-1.0` = open. In this task the hammer starts held, so the environment ignores this value and the fingers stay closed.
- `env.action_spec` returns `(low, high)` bound arrays


In [ ]:
low, high = env.action_spec
print("Action dimension:", low.shape[0])
print("Lower bounds:", np.round(low, 2))
print("Upper bounds:", np.round(high, 2))

print()
print("Position:  indices 0-2   (dx, dy, dz)")
print("Rotation:  indices 3-5   (droll, dpitch, dyaw)")
print("Gripper:   index 6       (+1=close, -1=open; ignored while holding)")


### 3.2 Info

`env.step(action)` also returns `info`. The person does not move.

| Key | What it does |
|---|---|
| `handle_pos` | Hammer handle center XYZ (meters) |
| `ellipsoid_pos` | Palm-sphere center XYZ (meters) |
| `hammer_speed` | Hammer linear speed (m/s) |
| `hold_steps` | Consecutive steps with the handle inside 2 cm and no faster than 5 mm/s |
| `body_contact` | 1 if the hammer or the arm touches the person outside the right hand |
| `hand_contact` | 1 if the hammer or the arm touches the right hand |
| `success` | 1 after 60 hold steps (3 seconds) |
| `failure` | 1 if the person was touched on this step |


## Question 3 — Info keys
**8 pts**

Use the example below as a reference and write your own answer in the next cell.

| Item | Pts |
|---|---|
| `info["hammer_speed"]` | 2 |
| `info["hold_steps"]` | 2 |
| `info["body_contact"]` | 2 |
| `info["hand_contact"]` | 2 |


In [ ]:
obs = env.reset()
print("handle_pos after reset:", obs["handle_pos"])

action = np.random.uniform(low, high)
obs, reward, done, info = env.step(action)
dist = np.linalg.norm(obs["handle_pos"] - info["ellipsoid_pos"])
print("handle_pos after 1 random step:", obs["handle_pos"])
print("ellipsoid_pos:                 ", info["ellipsoid_pos"])
print(f"distance: {dist:.4f} m")

print(f"reward = {reward:.4f}")
print(f"done   = {done}")
print(f"info keys: {list(info.keys())}")


In [ ]:
# ========== example ==========
print(f"success: {info.get('success', 0)}")
print(f"failure: {info.get('failure', 0)}")
# ==========================


In [ ]:
# --- write your answer from here ---

# -----------------------------------


## Question 4 — Random episode
**8 pts**

A window shows `agentview`. Click that window, then press `q` or `Esc` to close it.

Use the example below as a reference and write your own answer in the next cell. Print the eight info keys.

**Write one sentence per key** in the markdown cell after your code: briefly explain what the key means.

Write about these keys:

`success`  `failure`  `body_contact`  `hand_contact`  `handle_pos`  `ellipsoid_pos`  `hammer_speed`  `hold_steps`

| What to do | Pts |
|---|---|
| Print all eight info keys in the random episode | 3 |
| One sentence per key | 5 |


In [ ]:
def run_random_episode(print_keys, every=20):
    """Run env.horizon random steps. A window shows agentview.

    print_keys: names to print, from
    action, reward, distance, and the eight info keys
    (success, failure, body_contact, hand_contact,
    handle_pos, ellipsoid_pos, hammer_speed, hold_steps)
    """
    allowed = (
        "action",
        "reward",
        "distance",
        "success",
        "failure",
        "body_contact",
        "hand_contact",
        "handle_pos",
        "ellipsoid_pos",
        "hammer_speed",
        "hold_steps",
    )
    bad = [k for k in print_keys if k not in allowed]
    if bad:
        raise ValueError(f"unknown names {bad}. Choose from {allowed}")

    obs = env.reset()
    last = -1
    for t in range(env.horizon):
        action = np.random.uniform(low, high)
        obs, reward, done, info = env.step(action)
        last = t
        dist = float(np.linalg.norm(obs["handle_pos"] - info["ellipsoid_pos"]))
        values = {
            "action": np.round(action, 3),
            "reward": float(reward),
            "distance": round(dist, 4),
            "success": info.get("success", 0),
            "failure": info.get("failure", 0),
            "body_contact": info.get("body_contact", 0),
            "hand_contact": info.get("hand_contact", 0),
            "handle_pos": np.round(np.asarray(info["handle_pos"]), 4),
            "ellipsoid_pos": np.round(np.asarray(info["ellipsoid_pos"]), 4),
            "hammer_speed": float(info.get("hammer_speed", 0)),
            "hold_steps": info.get("hold_steps", 0),
        }
        if t % every == 0 or done:
            parts = [f"step {t:3d}"] + [f"{k}={values[k]}" for k in print_keys]
            print("  ".join(parts))
        try:
            cv2.imshow("random episode", obs["agentview_image"][::-1][..., ::-1])
            # q or Esc closes the window. The window must be focused.
            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                print("window closed (q / Esc)")
                break
        except Exception:
            pass
        if done:
            break
    try:
        cv2.destroyAllWindows()
        cv2.waitKey(1)
    except Exception:
        pass
    print(f"finished {last + 1} / {env.horizon} steps")


In [ ]:
# Close the window: click it, press q or Esc.
# ========== example ==========
run_random_episode(["action", "handle_pos"])
# ==========================


In [ ]:
# --- write your answer from here ---

# -----------------------------------


Write one sentence for each key below: briefly explain what the key means.

- `success`:
- `failure`:
- `body_contact`:
- `hand_contact`:
- `handle_pos`:
- `ellipsoid_pos`:
- `hammer_speed`:
- `hold_steps`:


---
## Part 4 — Reward Functions

The reward is the **learning signal** — it tells the RL algorithm what we want the robot to do.

### 4.1 The built-in sparse reward

With `reward_shaping=False` (current setting):
- `reward = 1.0` on the step that completes the 3 second hold
- `reward = 0.0` otherwise, including a step that touches the person

**Problem:** The arm almost never stays inside 2 cm at 5 mm/s for 3 seconds by chance, so the sparse reward provides almost no useful gradient for learning.

### 4.2 A dense reward

With `reward_shaping=True` the environment scores every step by distance only: 1 at the palm-sphere center, falling toward 0 farther away. It does not reward moving slowly or holding still. Touching the person sets `failure` and ends the episode.

Question 5 is a warmup on distance scores. Question 6 is your own reward. The next cell only plots the built-in dense reward under random actions, so you can see that it is not always zero.


In [ ]:
env_dense = suite.make(
    env_name="HumanRobotHandover",
    robots="XArm7Handover",
    use_camera_obs=False,
    has_offscreen_renderer=False,
    reward_shaping=True,           # <-- dense reward
    horizon=300,
    control_freq=20,
    seed=0,
    hard_reset=False,
)
env_dense.start_holding = True

low_d, high_d = env_dense.action_spec

obs_d = env_dense.reset()
dense_rewards = []
for _ in range(200):
    obs_d, r, done, _ = env_dense.step(np.random.uniform(low_d, high_d))
    dense_rewards.append(r)
    if done:
        obs_d = env_dense.reset()

env_dense.close()

plt.figure(figsize=(10, 3))
plt.plot(dense_rewards, label="dense reward")
plt.axhline(0, color="k", linewidth=0.5)
plt.xlabel("Step")
plt.ylabel("Reward")
plt.title("Dense reward during random exploration")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean dense reward: {np.mean(dense_rewards):.4f}")
print(f"Reward range: [{min(dense_rewards):.4f}, {max(dense_rewards):.4f}]")


## Question 5 — Distance score
**8 pts** (2 + 4 + 2)

### Question 5.1 — Score shapes
**2 pts**

Three one-line scores. `d` is distance in meters. Keep the score opposite to distance:

```python
score_1 = -d
score_2 = -d ** 2
score_3 = -d ** 3
```

Use the example below as a reference and write your own answer in the next cell. Add `score_2` and `score_3` the same way, and keep the red distance line.

| What to do | Pts |
|---|---|
| All three score curves, plus a red distance line, on one figure | 2 |


In [ ]:
d = np.linspace(0.35, 0.0, 200)
score_1 = -d
score_2 = -d ** 2
score_3 = -d ** 3


In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(d, d, color="red", label="distance")
# ========== example ==========
plt.plot(d, score_1, label="score_1")
# ==========================
plt.xlim(0.35, 0.0)
plt.xlabel("distance to sphere center (m)")
plt.ylabel("value")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(d, d, color="red", label="distance")
# --- write your answer from here ---

# -----------------------------------
plt.xlim(0.35, 0.0)
plt.xlabel("distance to sphere center (m)")
plt.ylabel("value")
plt.legend()
plt.tight_layout()
plt.show()


### Question 5.2 — Your score on a random episode
**6 pts** (4 + 2)

`d` is the handle-to-sphere distance at each step of a random episode. That array is already collected.

Use the example below as a reference and write your own answer in the next cell. Change the score line, in the same style as `score_1` / `score_2` / `score_3`.

**Write one sentence** in the cell after the plot: describe how the two lines change relative to each other during the episode.

| What to do | Pts |
|---|---|
| One-line score | 4 |
| One sentence | 2 |


In [ ]:
# Random episode. d is distance at each step.
obs = env.reset()
ds = []
for step in range(env.horizon):
    action = np.random.uniform(low, high)
    obs, reward, done, info = env.step(action)
    ds.append(np.linalg.norm(obs["handle_pos"] - info["ellipsoid_pos"]))
    if done:
        break
d = np.asarray(ds, dtype=float)

def minmax(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


In [ ]:
# ========== example ==========
score = -d
# ==========================

plt.figure(figsize=(8, 3))
plt.plot(minmax(d), label="distance (minmax)")
plt.plot(minmax(score), label="score (minmax)")
plt.xlabel("step")
plt.ylabel("normalized value")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- write your answer from here ---

# -----------------------------------

# already written
plt.figure(figsize=(8, 3))
plt.plot(minmax(d), label="distance (minmax)")
plt.plot(minmax(score), label="score (minmax)")
plt.xlabel("step")
plt.ylabel("normalized value")
plt.legend()
plt.tight_layout()
plt.show()


In one sentence, how do the two lines change relative to each other?

Your sentence:


## Question 6 — Write the reward
**50 pts** (20 + 15 + 15)

`prepare_reward(info)` is already written in the next cell: it gives `distance`, `speed`, `hold_steps`, and `distance_score`. After that cell is a list of what you can use. Write `custom_reward` in the cell after that list, below `# --- reward: write from here ---`. Do not use `obs`.

Every term you add should be zero or positive. Touching the person sets `info["failure"]` and ends the episode. **Last-step failure scores 0** — none of the three stages.

Numbers for this task:

- Palm sphere (the cyan ball) radius: **5 cm** (`0.05` m)
- Hold ball: **2 cm** (`0.02` m)
- Hold speed: **5 mm/s** (`0.005` m/s)
- Hold time: **60 steps** at 20 Hz, about **3 s**
- `distance` is handle center to palm-sphere center, meters
- `speed` is hammer linear speed, m/s
- `hold_steps` counts consecutive steps with distance ≤ 2 cm **and** speed ≤ 5 mm/s. If the handle is inside 2 cm but faster than 5 mm/s, `hold_steps` stays 0
- `distance_score = exp(-(distance / 0.15) ** 2)` is already computed for you

**Grading uses the last step of the eval episode only.** Going into 5 cm or 2 cm and then leaving does not count. If the last step is `failure`, the stage is 0. Part 6.2 prints the result as `Best final stage`; that line is your Question 6 stage.

The three stages are what a trained policy should do. The reward is marked on whether eval ends in that stage, not on copying a particular formula.

### Stage 1 — Finish inside the 5 cm sphere
**20 pts**

On the last step, `distance ≤ 0.05`. Mid-episode visits that leave again score 0 for this stage.

### Stage 2 — Finish inside the 2 cm ball
**15 pts**

On the last step, `distance ≤ 0.02`. This is position only. Speed does not matter. `hold_steps` is not the check here: the environment only increments `hold_steps` when the handle is also no faster than 5 mm/s.

### Stage 3 — Finish the 3 s hold
**15 pts**

On the last step, `hold_steps ≥ 60`. That already means the handle stayed inside 2 cm at no more than 5 mm/s for about 3 s. One frame inside 2 cm is not enough. If it leaves or speeds up, `hold_steps` goes back to 0.

| Stage | Last-step check | Pts |
|---|---|---|
| fail | `failure` | 0 |
| 1 | distance ≤ 5 cm | 20 |
| 2 | distance ≤ 2 cm | 15 |
| 3 | hold_steps ≥ 60: inside 2 cm and ≤ 5 mm/s for 3 s | 15 |


In [ ]:
def prepare_reward(info):
    """Read one step. This is preparation, not the reward students write.

    distance        Meters from the handle center to the palm-sphere center.
    speed           Hammer linear speed, meters per second.
    hold_steps      Steps in a row inside 2 cm at no more than 5 mm/s.
    distance_score  Reward for the corresponding distance.
    """
    distance = float(np.linalg.norm(info["handle_pos"] - info["ellipsoid_pos"]))
    speed = float(info["hammer_speed"])
    hold_steps = info["hold_steps"]

    distance_score = float(np.exp(-(distance / 0.15) ** 2))

    return distance, speed, hold_steps, distance_score


### What you can use in `custom_reward`

Write the function in the next cell. These are the quantities you can use. Do not use `obs`.

From `prepare_reward(info)`:

| Name | What it is |
|---|---|
| `distance` | Handle center to palm-sphere center, meters |
| `speed` | Hammer linear speed, m/s |
| `hold_steps` | Consecutive steps with distance ≤ 2 cm **and** speed ≤ 5 mm/s |
| `distance_score` | `exp(-(distance / 0.15) ** 2)`, already computed |

From `info` (same step):

| Key | What it is |
|---|---|
| `handle_pos` | Handle center (xyz) |
| `ellipsoid_pos` | Palm-sphere center (xyz) |
| `hammer_speed` | Same number as `speed` |
| `hold_steps` | Same number as `hold_steps` |
| `success` | 1 if the 3 s hold finished |
| `failure` | 1 if the robot touched the person |
| `body_contact` | 1 if contact with the person's body |
| `hand_contact` | 1 if contact with the person's hand |

You can also use `numpy` (`np`) and ordinary Python (`if`, `max`, `+`, `*`). Every term you add should be zero or positive.


In [ ]:
def custom_reward(info):
    """
    Use the values returned by prepare_reward() to write your own reward.
    Grading uses the last step only. Last-step failure scores 0.
    Stage 1: last step, distance <= 0.05.
    Stage 2: last step, distance <= 0.02.
    Stage 3: last step, hold_steps >= 60.
    """
    distance, speed, hold_steps, distance_score = prepare_reward(info)

    # --- reward: write from here ---

    # -----------------------------------


obs = env.reset()
obs, env_r, done, info = env.step(np.zeros(low.shape))
my_r = custom_reward(info)
distance, speed, hold_steps, distance_score = prepare_reward(info)
chosen = "  ".join(
    f"{cm:>2}cm {prepare_reward({'handle_pos': np.array([cm / 100.0, 0.0, 0.0]), 'ellipsoid_pos': np.zeros(3), 'hammer_speed': 0.0, 'hold_steps': 0})[3]:.3f}"
    for cm in (0, 5, 10, 20, 35, 70)
)
print("distance_score at 0, 5, 10, 20, 35, 70 cm")
print(f"  {chosen}")
print(f"env_reward    = {env_r:.4f}")
print(f"custom_reward = {my_r}")
print(f"  distance: {distance:.4f} m")
print(f"  speed:    {speed:.4f} m/s")
print(f"  hold_steps: {hold_steps}")
print(f"  distance_score: {distance_score:.3f}")


---
## Part 5 — Training an RL Policy

We use **Stable-Baselines3 (SB3)**, a reliable and well-documented RL library.

### 5.1 Gymnasium wrapper

SB3 expects the **Gymnasium** API:
- `obs, info = env.reset(seed=...)`
- `obs, reward, terminated, truncated, info = env.step(action)`

We write a thin wrapper that:
1. Converts robosuite's 4-return `step()` to Gymnasium's 5-return format
2. Flattens all non-image observations into a single vector
3. Applies our custom reward

The wrapper calls `suite.make` again. These are the knobs you already saw in Part 1:

| Parameter | What it does |
|---|---|
| `env_name` | Selects the task class |
| `robots` | Robot model. This task uses one arm, `XArm7Handover` |
| `use_camera_obs` | Adds camera images to the observation dict. Training turns this off |
| `horizon` | Episode ends after this many timesteps |
| `control_freq` | How many times per second the policy can issue actions |
| `reward_shaping` | `True` uses the dense handover reward instead of the sparse success reward |
| `start_holding` | Set on the environment after `suite.make`. The hammer starts in the gripper |


In [ ]:
_warned_empty_reward = False


def apply_reward_fn(reward_fn, info, env_reward=0.0):
    """Use the student reward. An empty custom_reward returns None and scores 0."""
    global _warned_empty_reward
    if reward_fn is None:
        return float(env_reward)
    out = reward_fn(info)
    if out is None:
        if not _warned_empty_reward:
            print("custom_reward returned None. Write it in Question 6; until then every reward is 0.")
            _warned_empty_reward = True
        return 0.0
    return float(out)

class HandoverGymEnv(gym.Env):
    """
    Gymnasium wrapper around robosuite HumanRobotHandover.

    - Flattens all non-image observations into a 1-D float32 vector
    - Plugs in a custom reward function
    - Compatible with Stable-Baselines3
    """

    def __init__(self, reward_fn=None, horizon=300):
        super().__init__()

        # Build the underlying robosuite environment (no cameras needed for training)
        self._env = suite.make(
            env_name="HumanRobotHandover",
            robots="XArm7Handover",
            use_camera_obs=False,
            has_offscreen_renderer=False,
            reward_shaping=True,   # replaced by custom_reward in step()
            horizon=horizon,
            control_freq=20,
            seed=0,
            hard_reset=False,
        )
        self._env.start_holding = True
        self._reward_fn = reward_fn
        self._low, self._high = self._env.action_spec

        # Figure out the observation shape by doing a dummy reset
        dummy_obs = self._env.reset()
        self._obs_keys = sorted(
            k for k in dummy_obs.keys() if not k.endswith("_image")
        )
        flat_obs = self._flatten_obs(dummy_obs)

        # Define Gymnasium spaces
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=flat_obs.shape, dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=self._low.astype(np.float32),
            high=self._high.astype(np.float32),
            dtype=np.float32,
        )

    def _flatten_obs(self, obs):
        """Concatenate selected obs keys into a float32 vector."""
        parts = [np.asarray(obs[k], dtype=np.float32).ravel() for k in self._obs_keys]
        return np.concatenate(parts)

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        obs = self._env.reset()
        return self._flatten_obs(obs), {}

    def step(self, action):
        obs, env_reward, done, info = self._env.step(action)

        # custom_reward(info) reads the step dict. obs is not an input.
        reward = apply_reward_fn(self._reward_fn, info, env_reward)

        # Success ends the episode. Touching the person is the other ending,
        # so that step is not treated as a timeout.
        terminated = bool(info.get("success", False)) or bool(info.get("failure", False))
        truncated = bool(done) and not terminated

        return self._flatten_obs(obs), float(reward), terminated, truncated, info

    def render(self):
        pass  # headless during training

    def close(self):
        self._env.close()

# Sanity check. Uses your custom_reward; an empty one gives reward 0.
_reward = custom_reward if "custom_reward" in dir() else None
gym_env = HandoverGymEnv(reward_fn=_reward)
obs_flat, _ = gym_env.reset(seed=0)
print(f"Observation vector dim: {obs_flat.shape[0]}")
print(f"Action space:           {gym_env.action_space}")

obs_flat, r, terminated, truncated, info = gym_env.step(gym_env.action_space.sample())
print(f"Step — reward: {r:.4f}  terminated: {terminated}  truncated: {truncated}")
gym_env.close()


### 5.2 Training with SAC

**Algorithms overview:**

| Algorithm | Type | Good for |
|---|---|---|
| **PPO** | On-policy | Stable, easy to tune; good default |
| **SAC** | Off-policy | More sample-efficient; works well for continuous control |
| **TD3** | Off-policy | Deterministic policy; robust for robotic control |

We use **SAC** — it tends to be more sample-efficient for robotic manipulation.

During training, a callback prints each eval episode's last-step stage every 1000 steps (5 episodes), then the counts for that eval (`stage0` / `stage1` / `stage2` / `stage3`). If no episode reached a stage, the totals are all `stage0`. The counts are written to `saved_models/handover_eval_stages.csv`.

There is one model save: every 5000 steps the same 5-episode eval is ranked by stage counts, in order `stage3`, then `stage2`, then `stage1`. More high-stage episodes wins. If that rank is better than before, it overwrites `saved_models/handover_best_sac/best_model.zip`. Starting a new training run deletes the old best first.

**Before a long run:** plug in the charger and keep the computer awake until training finishes. If the computer sleeps, training stops, and the run cannot be continued. Do not close the lid.

- Mac: in a terminal, run `caffeinate -i` and leave it open until training finishes. Press Ctrl+C there afterwards.
- Windows: Settings → System → Power & battery → Screen and sleep. Set "When plugged in, put my device to sleep after" to **Never**.
- Linux: in a terminal, run `systemd-inhibit --what=sleep sleep infinity` and leave it open until training finishes, or turn off automatic suspend in the power settings.


In [ ]:
Path("saved_models").mkdir(parents=True, exist_ok=True)
_reward = custom_reward if "custom_reward" in dir() else None
train_env = Monitor(
    HandoverGymEnv(reward_fn=_reward, horizon=300),
    filename=str(Path("saved_models") / "handover_train"),
)


def grade_final_stage(info):
    """Question 6 stage from the last step only. Mid-episode visits do not count."""
    dist = float(np.linalg.norm(np.asarray(info["handle_pos"]) - np.asarray(info["ellipsoid_pos"])))
    speed = float(info.get("hammer_speed", 0.0))
    hold = float(info.get("hold_steps", 0.0))
    failure = bool(info.get("failure", 0))
    if failure:
        return 0, "failed (touched the person)", dist, speed, hold, failure
    stage = 0
    if dist <= 0.05:
        stage = 1
    if dist <= 0.02:
        stage = 2
    if hold >= 60:
        stage = 3
    names = {
        0: "none",
        1: "stage 1 (finished inside 5 cm)",
        2: "stage 2 (finished inside 2 cm)",
        3: "stage 3 (finished 3 s hold)",
    }
    return stage, names[stage], dist, speed, hold, failure


def print_stage_report(label, info, reward=None):
    stage, name, dist, speed, hold, failure = grade_final_stage(info)
    extra = f"  reward={reward:.2f}" if reward is not None else ""
    print(f"{label}: FINAL {name}{extra}")
    print(
        f"         distance={dist:.4f} m  speed={speed:.4f} m/s  "
        f"hold_steps={hold:.0f}  failure={int(failure)}"
    )
    return stage


def append_stage_log(path, timestep, episode, info, reward):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    stage, _, dist, speed, hold, failure = grade_final_stage(info)
    new_file = not path.exists()
    with path.open("a") as f:
        if new_file:
            f.write("timestep,episode,stage,distance,speed,hold_steps,failure,reward\n")
        f.write(
            f"{timestep},{episode},{stage},{dist:.6f},{speed:.6f},{hold:.0f},{int(failure)},{float(reward):.6f}\n"
        )
    return stage


def stage_counts(stages):
    counts = [0, 0, 0, 0]
    for stage in stages:
        counts[int(stage)] += 1
    return counts


def print_stage_counts(label, stages):
    counts = stage_counts(stages)
    extra = "  (none reached a stage)" if counts[1] + counts[2] + counts[3] == 0 else ""
    print(
        f"{label}: stage0={counts[0]}  stage1={counts[1]}  "
        f"stage2={counts[2]}  stage3={counts[3]}{extra}"
    )
    return counts


def append_stage_counts(path, timestep, stages):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    counts = stage_counts(stages)
    new_file = not path.exists()
    with path.open("a") as f:
        if new_file:
            f.write("timestep,n_eval,stage0,stage1,stage2,stage3\n")
        f.write(
            f"{timestep},{len(stages)},{counts[0]},{counts[1]},{counts[2]},{counts[3]}\n"
        )
    return counts


def stage_rank_key(stages):
    """Higher stages first: more stage 3 beats more stage 2, then stage 1."""
    counts = stage_counts(stages)
    return (counts[3], counts[2], counts[1])


class StageEvalCallback(BaseCallback):
    """Eval last-step stages. Every 5000 steps, save if this batch is the best stage rank."""

    def __init__(
        self,
        eval_env,
        eval_freq=1000,
        n_eval_episodes=5,
        log_path=None,
        best_model_save_path=None,
        best_eval_freq=5000,
        verbose=1,
    ):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.log_path = Path(log_path) if log_path is not None else Path("saved_models") / "handover_eval_stages.csv"
        self.best_model_save_path = (
            Path(best_model_save_path)
            if best_model_save_path is not None
            else Path("saved_models") / "handover_best_sac"
        )
        self.best_eval_freq = best_eval_freq
        self.best_key = None

    def _on_training_start(self):
        if self.log_path.exists():
            self.log_path.unlink()
        # Part 6.1 loads the best model first, so a best from an earlier run must not survive.
        best_zip = self.best_model_save_path / "best_model.zip"
        if best_zip.exists():
            best_zip.unlink()
        self.best_key = None

    def _maybe_save_best(self, stages):
        if self.best_eval_freq <= 0 or self.num_timesteps % self.best_eval_freq != 0:
            return
        key = stage_rank_key(stages)
        if self.best_key is not None and key <= self.best_key:
            print(
                f"Best unchanged at timestep {self.num_timesteps}: "
                f"rank {key} (best {self.best_key})"
            )
            return
        self.best_key = key
        self.best_model_save_path.mkdir(parents=True, exist_ok=True)
        self.model.save(str(self.best_model_save_path / "best_model"))
        counts = stage_counts(stages)
        print(
            f"New best model at timestep {self.num_timesteps}: "
            f"stage0={counts[0]}  stage1={counts[1]}  "
            f"stage2={counts[2]}  stage3={counts[3]}  rank={key}"
        )

    def _on_step(self):
        if self.eval_freq <= 0 or self.n_calls % self.eval_freq != 0:
            return True
        print(f"\nEval stages at timestep {self.num_timesteps}")
        stages = []
        for ep in range(self.n_eval_episodes):
            obs, _ = self.eval_env.reset()
            ep_reward = 0.0
            info = {}
            for _ in range(300):
                action, _ = self.model.predict(obs, deterministic=True)
                obs, r, terminated, truncated, info = self.eval_env.step(action)
                ep_reward += float(r)
                if terminated or truncated:
                    break
            stages.append(print_stage_report(f"  eval ep {ep + 1}", info, reward=ep_reward))
        print_stage_counts(f"Eval totals at timestep {self.num_timesteps}", stages)
        append_stage_counts(self.log_path, self.num_timesteps, stages)
        self._maybe_save_best(stages)
        return True


stage_eval_env = HandoverGymEnv(reward_fn=_reward, horizon=300)
stage_callback = StageEvalCallback(
    eval_env=stage_eval_env,
    eval_freq=1000,
    n_eval_episodes=5,
    log_path=Path("saved_models") / "handover_eval_stages.csv",
    best_model_save_path=Path("saved_models") / "handover_best_sac",
    best_eval_freq=5000,
)

# SAC model
model = SAC(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    buffer_size=200_000,
    batch_size=256,
    train_freq=1,
    gradient_steps=1,
    learning_starts=1000,
    ent_coef=0.005,
    verbose=1,
    seed=0,
    device="cpu",   # change to "mps" on Apple Silicon or "cuda" on Nvidia GPU
)

print("Policy architecture:")
print(model.policy)


In [ ]:
# ==============================================================
# TRAIN
# Assignment length. Keep the computer plugged in and awake
# until model.learn finishes. On the Mac we tested, this
# took about 75 minutes.
# ==============================================================
TOTAL_TIMESTEPS = 120_000

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[stage_callback],
    progress_bar=False,
)

model.save("saved_models/sac_handover_notebook")
print("Training complete. Model saved to saved_models/sac_handover_notebook.zip")
print("Eval stages written to saved_models/handover_eval_stages.csv")

train_env.close()
stage_eval_env.close()


### 5.3 Understanding the training output

SB3 prints a progress table every N updates:

```
| rollout/           |          |
|    ep_len_mean     | 300      |  <- average episode length
|    ep_rew_mean     | 1.23     |  <- average episode reward (your custom_reward)
| time/              |          |
|    episodes        | 4        |  <- finished episodes so far
|    fps             | 59       |  <- environment steps per second
|    time_elapsed    | 20       |  <- wall-clock seconds so far
|    total_timesteps | 1200     |  <- environment steps so far
| train/             |          |
|    actor_loss      | -0.456   |  <- policy gradient loss
|    critic_loss     | 0.012    |  <- value function MSE loss
|    ent_coef        | 0.234    |  <- entropy coefficient (exploration)
|    learning_rate   | 0.0003   |  <- optimizer learning rate
|    n_updates       | 199      |  <- gradient updates so far
```

**What to watch:** `ep_rew_mean` should trend upward over time. Each eval also prints last-step stages, then the counts for that eval (`stage0` / `stage1` / `stage2` / `stage3`). If no episode reached stage 1 or higher, the totals are all in `stage0`. The counts are written to `saved_models/handover_eval_stages.csv`.

### 5.4 Learning curve


In [ ]:
monitor_path = Path("saved_models") / "handover_train.monitor.csv"
if monitor_path.exists():
    df = pd.read_csv(monitor_path, skiprows=1)
    plt.figure(figsize=(10, 4))
    plt.plot(df.index, df["r"].rolling(20, min_periods=1).mean(),
             label="Episode reward (rolling mean, window=20)")
    plt.xlabel("Episode")
    plt.ylabel("Episode reward")
    plt.title("Learning Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No monitor CSV found yet — run training first.")

stage_path = Path("saved_models") / "handover_eval_stages.csv"
if stage_path.exists():
    sdf = pd.read_csv(stage_path)
    plt.figure(figsize=(10, 4))
    for k, label in ((0, "stage 0"), (1, "stage 1"), (2, "stage 2"), (3, "stage 3")):
        plt.plot(sdf["timestep"], sdf[f"stage{k}"], marker="o", label=label)
    plt.xlabel("Timestep")
    plt.ylabel("Eval episodes")
    plt.title("Eval last-step stage counts during training")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No stage CSV found yet — run training first.")


---
## Part 6 — Evaluating and Visualizing the Trained Policy

### 6.1 Loading a saved model


In [ ]:
best_model_path  = Path("saved_models/handover_best_sac/best_model.zip")
final_model_path = Path("saved_models/sac_handover_notebook.zip")

if best_model_path.exists():
    loaded_model = SAC.load(str(best_model_path))
    print("Loaded best model:", best_model_path)
elif final_model_path.exists():
    loaded_model = SAC.load(str(final_model_path))
    print("Loaded final model:", final_model_path)
else:
    print("No saved model found. Please run Part 5 training first.")
    loaded_model = None


### 6.2 Evaluate over multiple episodes

Each episode is graded from its **last step** (Question 6). Run **10** eval episodes. After all of them, the printout shows how many ended in each stage. If none reached stage 1 or higher, the totals are all `stage0`. The counts are written to `saved_models/handover_final_eval_stages.csv`.


In [ ]:
if loaded_model is not None:
    _reward = custom_reward if "custom_reward" in dir() else None
    eval_wrapped = HandoverGymEnv(reward_fn=_reward, horizon=300)
    n_episodes = 10
    ep_rewards = []
    ep_stages = []
    final_stage_path = Path("saved_models") / "handover_final_eval_stages.csv"
    if final_stage_path.exists():
        final_stage_path.unlink()

    for ep in range(n_episodes):
        obs, _ = eval_wrapped.reset(seed=ep)
        ep_reward = 0.0
        info = {}
        for _ in range(300):
            action, _ = loaded_model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, info = eval_wrapped.step(action)
            ep_reward += r
            if terminated or truncated:
                break
        ep_rewards.append(ep_reward)
        ep_stages.append(print_stage_report(f"Episode {ep+1}", info, reward=ep_reward))

    eval_wrapped.close()
    print()
    print(f"Mean reward:     {np.mean(ep_rewards):.2f}")
    print_stage_counts("Final eval totals", ep_stages)
    append_stage_counts(final_stage_path, "final", ep_stages)
    print(f"Best final stage: {max(ep_stages) if ep_stages else 0}")
    print(f"Stages: {ep_stages}")
    print(f"Wrote {final_stage_path}")
else:
    print("No saved model found. Run Part 5 first.")


### 6.3 Recording an evaluation video

Watch `trained_policy.mp4`. The last-step stage is printed with the file and written to `saved_models/handover_video_eval_stage.csv`.


In [ ]:
if loaded_model is not None:

    # Create a camera-enabled env for recording
    video_env = suite.make(
        env_name="HumanRobotHandover",
        robots="XArm7Handover",
        use_camera_obs=True,
        camera_names="agentview",
        reward_shaping=True,
        horizon=300,
        control_freq=20,
        seed=0,
        hard_reset=False,
    )
    video_env.start_holding = True

    # We need the same obs keys as used during training (no images)
    _dummy = video_env.reset()
    _obs_keys = sorted(k for k in _dummy.keys() if not k.endswith("_image"))

    def flatten_for_model(raw_obs):
        parts = [np.asarray(raw_obs[k], dtype=np.float32).ravel() for k in _obs_keys]
        return np.concatenate(parts)

    raw_obs = video_env.reset()
    frames = [raw_obs["agentview_image"][::-1]]
    total_reward = 0.0
    final_info = {}

    for step in range(300):
        flat_obs = flatten_for_model(raw_obs)
        action, _ = loaded_model.predict(flat_obs, deterministic=True)
        raw_obs, r, done, final_info = video_env.step(action)
        frames.append(raw_obs["agentview_image"][::-1])
        _reward = custom_reward if "custom_reward" in dir() else None
        total_reward += apply_reward_fn(_reward, final_info, r)
        if done:
            break

    video_env.close()

    with imageio.get_writer("trained_policy.mp4", fps=20) as writer:
        for f in frames:
            writer.append_data(f)

    print(f"Saved trained_policy.mp4  ({len(frames)} frames)")
    print(f"Total reward: {total_reward:.2f}")
    print_stage_report("Video episode", final_info, reward=total_reward)
    video_stage_path = Path("saved_models") / "handover_video_eval_stage.csv"
    if video_stage_path.exists():
        video_stage_path.unlink()
    video_stage = grade_final_stage(final_info)[0]
    print_stage_counts("Video eval totals", [video_stage])
    append_stage_counts(video_stage_path, "video", [video_stage])

    display(Video("trained_policy.mp4", embed=True, width=400))
